# API 매칭 — 알약 118종 정보 수집

식약처 공식 DB + AI Hub 원본 JSON을 활용해서 학습 데이터 118종의 효능/용법/주의사항을 수집합니다.

**사용 API**
- 낱알식별 API (15057639)
- e약은요 API (15075057)
- 의약품 제품 허가정보 API (15095677)

**산출물**
- `pill_db_nalal.csv` — 낱알식별 DB (25,508건)
- `pill_db_eyak.csv` — e약은요 DB (4,704건)
- `pill_info_master.json` — 118종 통합 마스터

In [ ]:
import os
import re
import json
import time
import requests
import pandas as pd
from pathlib import Path

# 인증키 (3개 API 동일) — 환경변수로 관리
KEY = os.environ.get("DATA_GO_KR_KEY", "")
if not KEY:
    raise RuntimeError(
        "환경변수 DATA_GO_KR_KEY 설정 필요\n"
        '  PowerShell: $env:DATA_GO_KR_KEY = "발급받은_키"\n'
        '  Bash:       export DATA_GO_KR_KEY="발급받은_키"'
    )

# 출력 경로
OUT_DIR = Path(r"D:\pill_detection\eda\api_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR:", OUT_DIR)
print("존재:", OUT_DIR.exists())

OUT_DIR: D:\pill_detection\eda\api_data
존재: True


## STEP 1 — 식약처 두 DB 일괄 수집

낱알식별 API와 e약은요 API에서 전체 DB를 페이지 단위로 받아옵니다. 매칭에 직접 쓰는 건 STEP 2~3이지만, 추후 검색/검증용으로 로컬 캐시해둡니다.

- 낱알식별: 25,508건 / 256페이지 (≈1분 20초)
- e약은요: 4,704건 / 48페이지 (≈20초)

In [2]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

URL_NALAL = "https://apis.data.go.kr/1471000/MdcinGrnIdntfcInfoService03/getMdcinGrnIdntfcInfoList03"
ROWS = 100

# 1) totalCount 확인
params0 = {"serviceKey": KEY, "type": "json", "numOfRows": 1, "pageNo": 1}
r0 = requests.get(URL_NALAL, params=params0, timeout=10)
total = r0.json()['body']['totalCount']
n_pages = (total + ROWS - 1) // ROWS
print(f"낱알식별 전체: {total:,}건 / {n_pages}페이지")

# 2) 한 페이지 fetch
def fetch_page_nalal(page):
    params = {"serviceKey": KEY, "type": "json", "numOfRows": ROWS, "pageNo": page}
    try:
        r = requests.get(URL_NALAL, params=params, timeout=15)
        items = (r.json().get('body') or {}).get('items') or []
        return page, items, None
    except Exception as e:
        return page, [], str(e)

# 3) 병렬 수집
all_items, failed = [], []
with ThreadPoolExecutor(max_workers=5) as ex:
    futures = {ex.submit(fetch_page_nalal, p): p for p in range(1, n_pages + 1)}
    for fut in tqdm(as_completed(futures), total=n_pages, desc="낱알식별"):
        page, items, err = fut.result()
        if err:
            failed.append((page, err))
        else:
            all_items.extend(items)

# 4) 실패 페이지 재시도
if failed:
    print(f"\n실패 페이지 {len(failed)}개 재시도...")
    for page, _ in failed:
        _, items, err = fetch_page_nalal(page)
        if not err:
            all_items.extend(items)
        time.sleep(0.3)

# 5) 저장
db_nalal = pd.DataFrame(all_items)
out_path = OUT_DIR / "pill_db_nalal.csv"
db_nalal.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"\n수집: {len(all_items):,}건")
print(f"저장: {out_path}")
print(f"shape: {db_nalal.shape}")

낱알식별 전체: 25,508건 / 256페이지


낱알식별: 100%|██████████| 256/256 [01:02<00:00,  4.12it/s]



수집: 25,508건
저장: D:\pill_detection\eda\api_data\pill_db_nalal.csv
shape: (25508, 33)


In [3]:
URL_EYAK = "https://apis.data.go.kr/1471000/DrbEasyDrugInfoService/getDrbEasyDrugList"
ROWS = 100

# 1) totalCount 확인
params0 = {"serviceKey": KEY, "type": "json", "numOfRows": 1, "pageNo": 1}
r0 = requests.get(URL_EYAK, params=params0, timeout=10)
total = r0.json()['body']['totalCount']
n_pages = (total + ROWS - 1) // ROWS
print(f"e약은요 전체: {total:,}건 / {n_pages}페이지")

# 2) 한 페이지 fetch
def fetch_page_eyak(page):
    params = {"serviceKey": KEY, "type": "json", "numOfRows": ROWS, "pageNo": page}
    try:
        r = requests.get(URL_EYAK, params=params, timeout=15)
        items = (r.json().get('body') or {}).get('items') or []
        return page, items, None
    except Exception as e:
        return page, [], str(e)

# 3) 병렬 수집
all_items, failed = [], []
with ThreadPoolExecutor(max_workers=5) as ex:
    futures = {ex.submit(fetch_page_eyak, p): p for p in range(1, n_pages + 1)}
    for fut in tqdm(as_completed(futures), total=n_pages, desc="e약은요"):
        page, items, err = fut.result()
        if err:
            failed.append((page, err))
        else:
            all_items.extend(items)

# 4) 실패 페이지 재시도
if failed:
    print(f"\n실패 페이지 {len(failed)}개 재시도...")
    for page, _ in failed:
        _, items, err = fetch_page_eyak(page)
        if not err:
            all_items.extend(items)
        time.sleep(0.3)

# 5) 저장
db_eyak = pd.DataFrame(all_items)
out_path = OUT_DIR / "pill_db_eyak.csv"
db_eyak.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"\n수집: {len(all_items):,}건")
print(f"저장: {out_path}")
print(f"shape: {db_eyak.shape}")

e약은요 전체: 4,704건 / 48페이지


e약은요: 100%|██████████| 48/48 [00:15<00:00,  3.19it/s]



수집: 4,704건
저장: D:\pill_detection\eda\api_data\pill_db_eyak.csv
shape: (4704, 14)


## STEP 2 — AI Hub 원본 JSON에서 ITEM_SEQ 추출

K-code 폴더의 JSON에 식약처 `item_seq` 필드가 포함되어 있어 시각/이름 매칭 없이 바로 확보할 수 있습니다.

1. clean_df 로드 → 118종 agg
2. 118개 K-code 폴더에서 첫 JSON 1개씩 → item_seq 추출
3. agg와 left join + 일양하이트린정 보정

In [4]:
CLEAN_DF = r"D:\pill_detection\data\team_preprocessed\clean_df.csv"
df = pd.read_csv(CLEAN_DF, low_memory=False)

# dl_name 기준 통합 (NaN 아닌 값 우선)
agg = (
    df.groupby('dl_name')
      .agg({
          'dl_mapping_code': lambda x: x.dropna().iloc[0] if x.notna().any() else None,
          'dl_company':      lambda x: x.dropna().iloc[0] if x.notna().any() else None,
          'source':          lambda x: 'kaggle' if x.isna().any() and x.notna().any()
                                       else ('aihub_only' if x.notna().all() else 'kaggle_only'),
      })
      .reset_index()
      .sort_values('dl_name')
      .reset_index(drop=True)
)
agg.insert(0, 'class_idx', range(len(agg)))

print(f"전체 ann: {len(df):,}")
print(f"통합 클래스 수: {len(agg)}종")
print(f"\nsource 분포:")
print(agg['source'].value_counts())
print(f"\nK-code 있음: {agg['dl_mapping_code'].notna().sum()} / 없음: {agg['dl_mapping_code'].isna().sum()}")
print(f"회사명 있음: {agg['dl_company'].notna().sum()} / 없음: {agg['dl_company'].isna().sum()}")

전체 ann: 39,864
통합 클래스 수: 118종

source 분포:
source
aihub_only     62
kaggle         54
kaggle_only     2
Name: count, dtype: int64

K-code 있음: 116 / 없음: 2
회사명 있음: 116 / 없음: 2


In [5]:
import glob
from pathlib import Path

DATA_ROOT = Path(r"D:\pill_detection\data")
TL_DIRS = [DATA_ROOT / f"TL_{n}_조합" for n in [1, 3, 4, 5, 6, 7, 8]]

# K-code별 첫 JSON 1개씩 (각 K-code 폴더의 첫 파일)
seen_kcode = {}
for tl_dir in TL_DIRS:
    if not tl_dir.exists():
        continue
    for kdir in tl_dir.glob("*_json/K-*"):
        kcode = kdir.name
        if kcode in seen_kcode:
            continue
        jsons = list(kdir.glob("*.json"))
        if jsons:
            seen_kcode[kcode] = jsons[0]

print(f"K-code 폴더 수: {len(seen_kcode)}")

# 각 JSON에서 dl_name + item_seq 추출
records = []
for kcode, jpath in tqdm(seen_kcode.items(), desc="JSON 파싱"):
    try:
        with open(jpath, encoding='utf-8') as f:
            raw = json.load(f)
        img = raw.get('images', [{}])[0]
        records.append({
            'K_code': kcode,
            'dl_idx': img.get('dl_idx'),
            'dl_name': img.get('dl_name'),
            'dl_company': img.get('dl_company'),
            'item_seq': img.get('item_seq'),
            'di_edi_code': img.get('di_edi_code'),
            'di_class_no': img.get('di_class_no'),
        })
    except Exception as e:
        print(f"실패 {kcode}: {e}")

mapping_df = pd.DataFrame(records)
print(f"\n수집: {len(mapping_df)}건")
print(f"item_seq 있음: {mapping_df['item_seq'].notna().sum()}")
print(f"dl_name unique: {mapping_df['dl_name'].nunique()}종")

K-code 폴더 수: 118


JSON 파싱: 100%|██████████| 118/118 [00:00<00:00, 3796.51it/s]


수집: 118건
item_seq 있음: 118
dl_name unique: 116종


In [6]:
# 1) -오류 폴더 제외
mapping_clean = mapping_df[~mapping_df['K_code'].str.contains('오류')].copy()
print(f"clean: {len(mapping_clean)}종 (dl_name unique: {mapping_clean['dl_name'].nunique()})")

# 2) agg(118종)와 dl_name 기준 left join
final_mapping = agg.merge(
    mapping_clean[['dl_name', 'item_seq', 'di_edi_code', 'di_class_no']],
    on='dl_name',
    how='left'
)

print(f"\nleft join 후 ITEM_SEQ 보유: {final_mapping['item_seq'].notna().sum()}/118")
print(f"미확보:")
print(final_mapping[final_mapping['item_seq'].isna()][['class_idx', 'dl_name', 'source']].to_string(index=False))

# 3) 일양하이트린정 2mg 수동 보정 (식약처 검색 정확 일치)
final_mapping.loc[final_mapping['class_idx'] == 83, 'item_seq'] = 199101125

# 4) 기넥신에프정 수출용 — 식약처 일반 등록 없음, 추후 STEP 4에서 40mg 정으로 매핑
print(f"\n보정 후 ITEM_SEQ 보유: {final_mapping['item_seq'].notna().sum()}/118")
print(f"최종 미확보: {final_mapping[final_mapping['item_seq'].isna()]['dl_name'].tolist()}")

clean: 116종 (dl_name unique: 116)

left join 후 ITEM_SEQ 보유: 116/118
미확보:
 class_idx            dl_name      source
         4 기넥신에프정(은행엽엑스)(수출용) kaggle_only
        83        일양하이트린정 2mg kaggle_only

보정 후 ITEM_SEQ 보유: 117/118
최종 미확보: ['기넥신에프정(은행엽엑스)(수출용)']


## STEP 3 — 의약품 제품 허가정보 API로 효능 본문 수집

`getDrugPrdtPrmsnDtlInq06` 메서드에 `EE_DOC_DATA`(효능) / `UD_DOC_DATA`(용법) / `NB_DOC_DATA`(주의사항) 필드가 XML 형식으로 들어있습니다.

XML 패턴 두 가지를 모두 처리해야 합니다:
- `<ARTICLE><PARAGRAPH><![CDATA[본문]]></PARAGRAPH></ARTICLE>`
- `<ARTICLE title="본문" />` (self-closing, title 속성이 본문)

폐업/취하 의약품은 식약처 API가 응답하지 않아 STEP 4에서 nedrug 사이트 직접 크롤링으로 보강합니다.

In [7]:
URL_DTL = "https://apis.data.go.kr/1471000/DrugPrdtPrmsnInfoService07/getDrugPrdtPrmsnDtlInq06"

# 117종 ITEM_SEQ 추출
seqs = final_mapping[final_mapping['item_seq'].notna()].copy()
seqs['item_seq_str'] = seqs['item_seq'].apply(lambda x: str(int(x)))
print(f"수집 대상: {len(seqs)}종")

def fetch_dtl(item_seq):
    params = {
        "serviceKey": KEY, "type": "json",
        "numOfRows": 1, "pageNo": 1,
        "item_seq": item_seq,
    }
    try:
        r = requests.get(URL_DTL, params=params, timeout=15)
        items = (r.json().get('body') or {}).get('items') or []
        if items:
            return item_seq, items[0], None
        return item_seq, None, "no_data"
    except Exception as e:
        return item_seq, None, str(e)

# 병렬 수집
results_dtl = {}
failed_dtl = []
with ThreadPoolExecutor(max_workers=5) as ex:
    futures = {ex.submit(fetch_dtl, s): s for s in seqs['item_seq_str']}
    for fut in tqdm(as_completed(futures), total=len(seqs), desc="허가정보 상세"):
        seq, data, err = fut.result()
        if err or data is None:
            failed_dtl.append((seq, err))
        else:
            results_dtl[seq] = data

print(f"\n수집: {len(results_dtl)}/{len(seqs)} | 실패: {len(failed_dtl)}")
if failed_dtl:
    print("실패 SEQ:")
    for seq, err in failed_dtl:
        info = final_mapping[final_mapping['item_seq'] == int(seq)][['class_idx', 'dl_name']].iloc[0]
        print(f"  {seq} ({info['dl_name']}) — {err}")

수집 대상: 117종


허가정보 상세: 100%|██████████| 117/117 [00:32<00:00,  3.58it/s]


수집: 111/117 | 실패: 6
실패 SEQ:
  200500251 (가바토파정 100mg) — no_data
  200611524 (마도파정) — no_data
  201800568 (바이오탑디캡슐 50mg/병) — no_data
  201006967 (울트라셋이알서방정) — no_data
  199603003 (타이레놀이알서방정(아세트아미노펜)(수출용)) — no_data
  199303108 (타이레놀정500mg) — no_data


In [8]:
import xml.etree.ElementTree as ET
import html as _html

def parse_doc_xml(xml_str):
    """허가정보 EE/UD/NB_DOC_DATA XML을 평문으로 변환.
    
    두 패턴 처리:
      1) <ARTICLE><PARAGRAPH><![CDATA[본문]]></PARAGRAPH></ARTICLE>
      2) <ARTICLE title="본문" /> (self-closing, title 속성이 본문)
    """
    if not xml_str:
        return ""
    
    try:
        root = ET.fromstring(xml_str)
    except ET.ParseError:
        # XML 파싱 실패 → CDATA만이라도 추출
        paragraphs = re.findall(r'<!\[CDATA\[(.*?)\]\]>', xml_str, re.DOTALL)
        cleaned = [_html.unescape(p).strip() for p in paragraphs]
        return '\n'.join(p for p in cleaned if p and p != '\xa0')
    
    lines = []
    for article in root.iter('ARTICLE'):
        title = article.get('title', '').strip()
        if title:
            lines.append(title)
        for para in article.iter('PARAGRAPH'):
            text = (para.text or '').strip()
            if text and text != '\xa0':
                lines.append(_html.unescape(text))
    
    return '\n'.join(lines)


# dtl_df 빌드
dtl_records = []
for seq, item in results_dtl.items():
    dtl_records.append({
        'item_seq': seq,
        'efficacy': parse_doc_xml(item.get('EE_DOC_DATA', '')),
        'usage':    parse_doc_xml(item.get('UD_DOC_DATA', '')),
        'caution':  parse_doc_xml(item.get('NB_DOC_DATA', '')),
        'storage':  item.get('STORAGE_METHOD'),
        'chart':    item.get('CHART'),
        'material': item.get('MATERIAL_NAME'),
        'etc_otc':  item.get('ETC_OTC_CODE'),
        'image_url': item.get('BIG_PRDT_IMG_URL'),
    })

dtl_df = pd.DataFrame(dtl_records)
print(f"dtl_df: {len(dtl_df)}종")
print(f"  효능 있음: {dtl_df['efficacy'].apply(bool).sum()}")
print(f"  용법 있음: {dtl_df['usage'].apply(bool).sum()}")
print(f"  주의 있음: {dtl_df['caution'].apply(bool).sum()}")

# 두 패턴 샘플 검증
print("\n--- 게보린정 (CDATA 패턴) 효능 샘플 ---")
print(dtl_df[dtl_df['item_seq'] == '197900277']['efficacy'].iloc[0][:200])
print("\n--- 놀텍정 (self-closing 패턴) 효능 샘플 ---")
print(dtl_df[dtl_df['item_seq'] == '200810688']['efficacy'].iloc[0][:200])

dtl_df: 111종
  효능 있음: 111
  용법 있음: 111
  주의 있음: 111

--- 게보린정 (CDATA 패턴) 효능 샘플 ---
 
다음 질환의 진통 및 해열시 단기치료:
- 두통, 치통, 발치(이를 뽑음)후 동통(통증), 인후(목구멍)통, 귀의 통증, 관절통, 신경통, 요(허리)통, 근육통, 견통(어깨통증), 타박통, 골절통, 염좌통(삔 통증), 월경통(생리통), 외상(상처)통의 진통
- 오한(춥고 떨리는 증상), 발열시의 해열

--- 놀텍정 (self-closing 패턴) 효능 샘플 ---
1. 십이지장궤양의 단기치료
2. 위궤양의 단기치료
3. 미란성식도염의 단기치료
4. 헬리코박터필로리에 감염된 위·십이지장궤양의 재발 방지를 위한 항생제 병용요법


## STEP 4 — 수출용 1종 + 폐업 6종 보강

STEP 3에서 누락된 7종 보강:
- **수출용 1종** (기넥신에프정 수출용): 식약처 일반 등록 없음 → 같은 성분의 일반정 SEQ로 매핑
- **폐업 6종**: 식약처 API는 폐업/취하 의약품 응답 안 함 → nedrug 사이트 직접 크롤링

In [9]:
# 수출용 → 일반정 SEQ로 매핑 (성분 동일: 은행엽건조엑스)
SEQ_GINEXIN = '199102014'

seq, data, err = fetch_dtl(SEQ_GINEXIN)
if err:
    raise RuntimeError(f"기넥신 fetch 실패: {err}")
results_dtl[SEQ_GINEXIN] = data

# final_mapping의 class_idx=4 item_seq 보정
final_mapping.loc[final_mapping['class_idx'] == 4, 'item_seq'] = int(SEQ_GINEXIN)

print(f"ITEM_NAME: {data['ITEM_NAME']}")
print(f"ENTP_NAME: {data['ENTP_NAME']}")
print(f"성분: {data['MATERIAL_NAME']}")
print(f"효능 첫줄: {parse_doc_xml(data['EE_DOC_DATA'])[:100]}")

print(f"\nresults_dtl 크기: {len(results_dtl)}")
print(f"final_mapping ITEM_SEQ 보유: {final_mapping['item_seq'].notna().sum()}/118")

ITEM_NAME: 기넥신에프정40밀리그램(은행엽건조엑스)
ENTP_NAME: 에스케이케미칼(주)
성분: 총량 : 이 약 1정(253.13 mg) 중|성분명 : 은행엽건조엑스|분량 : 40.0|단위 : 밀리그램|규격 : 생규|성분정보 : |비고 :
효능 첫줄: 1. 주효능효과
① 말초동맥 순환장애(간헐성 파행증)의 치료
② 어지러움, 혈관성 및 퇴행성 이명(귀울림)
③ 이명(귀울림), 두통, 기억력감퇴, 집중력장애, 우울감, 어지러움 등

results_dtl 크기: 112
final_mapping ITEM_SEQ 보유: 118/118


In [10]:
from bs4 import BeautifulSoup

URL_NEDRUG = "https://nedrug.mfds.go.kr/pbp/CCBBB01/getItemDetail"
HEADERS = {"User-Agent": "Mozilla/5.0"}

def extract_nedrug_section(soup, header_class):
    """효능/용법/주의 섹션 본문 추출 (header div 다음 형제 info_box)."""
    header_div = soup.find('div', class_=header_class)
    if not header_div:
        return ""
    info_box = header_div.find_next_sibling('div', class_='info_box')
    if not info_box:
        return ""
    return info_box.get_text(separator='\n', strip=True)

# STEP 3 실패 6종 자동 추출
failed_specs = []
for seq, _ in failed_dtl:
    info = final_mapping[final_mapping['item_seq'].astype('Int64') == int(seq)].iloc[0]
    failed_specs.append((int(info['class_idx']), info['dl_name'], seq))

print("폐업/취하 6종:")
for cidx, name, seq in failed_specs:
    print(f"  [{cidx:3d}] {name} ({seq})")

# 크롤링
nedrug_results = {}
for cidx, name, seq in tqdm(failed_specs, desc="nedrug 크롤링"):
    try:
        r = requests.get(URL_NEDRUG, params={"itemSeq": seq}, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(r.text, 'html.parser')
        nedrug_results[cidx] = {
            'item_seq': seq,
            'efficacy': extract_nedrug_section(soup, '_preview_ee'),
            'usage':    extract_nedrug_section(soup, '_preview_ud'),
            'caution':  extract_nedrug_section(soup, '_preview_nb'),
        }
        time.sleep(0.3)
    except Exception as e:
        print(f"  [{cidx}] {name}: 실패 - {e}")

# 검증
print(f"\n수집: {len(nedrug_results)}/6")
for cidx, data in nedrug_results.items():
    name = next(n for c, n, _ in failed_specs if c == cidx)
    print(f"  [{cidx:3d}] {name}: 효능 {len(data['efficacy'])}자 / 용법 {len(data['usage'])}자 / 주의 {len(data['caution'])}자")

폐업/취하 6종:
  [  0] 가바토파정 100mg (200500251)
  [ 32] 마도파정 (200611524)
  [ 39] 바이오탑디캡슐 50mg/병 (201800568)
  [ 78] 울트라셋이알서방정 (201006967)
  [104] 타이레놀이알서방정(아세트아미노펜)(수출용) (199603003)
  [105] 타이레놀정500mg (199303108)


nedrug 크롤링:  17%|█▋        | 1/6 [00:00<00:03,  1.65it/s]

  [0] 가바토파정 100mg: 실패 - ('Connection aborted.', ConnectionResetError(10054, '현재 연결은 원격 호스트에 의해 강제로 끊겼습니다', None, 10054, None))


nedrug 크롤링:  67%|██████▋   | 4/6 [00:05<00:02,  1.24s/it]

  [78] 울트라셋이알서방정: 실패 - ('Connection aborted.', ConnectionResetError(10054, '현재 연결은 원격 호스트에 의해 강제로 끊겼습니다', None, 10054, None))


nedrug 크롤링: 100%|██████████| 6/6 [00:09<00:00,  1.59s/it]


수집: 4/6
  [ 32] 마도파정: 효능 22자 / 용법 905자 / 주의 5568자
  [ 39] 바이오탑디캡슐 50mg/병: 효능 27자 / 용법 46자 / 주의 572자
  [104] 타이레놀이알서방정(아세트아미노펜)(수출용): 효능 55자 / 용법 52자 / 주의 3147자
  [105] 타이레놀정500mg: 효능 100자 / 용법 118자 / 주의 3087자


In [11]:
# 누락 cidx 자동 추출
missing = [(c, n, s) for c, n, s in failed_specs if c not in nedrug_results]
print(f"재시도 대상: {len(missing)}종")
for c, n, s in missing:
    print(f"  [{c}] {n} ({s})")

# 재시도 (간격 늘림)
for cidx, name, seq in missing:
    for attempt in range(3):
        try:
            time.sleep(1.0)
            r = requests.get(URL_NEDRUG, params={"itemSeq": seq}, headers=HEADERS, timeout=20)
            soup = BeautifulSoup(r.text, 'html.parser')
            nedrug_results[cidx] = {
                'item_seq': seq,
                'efficacy': extract_nedrug_section(soup, '_preview_ee'),
                'usage':    extract_nedrug_section(soup, '_preview_ud'),
                'caution':  extract_nedrug_section(soup, '_preview_nb'),
            }
            print(f"  [{cidx}] {name}: 효능 {len(nedrug_results[cidx]['efficacy'])}자 (시도 {attempt+1})")
            break
        except Exception as e:
            print(f"  [{cidx}] {name}: 시도 {attempt+1} 실패 - {e}")

print(f"\n최종 수집: {len(nedrug_results)}/6")

재시도 대상: 2종
  [0] 가바토파정 100mg (200500251)
  [78] 울트라셋이알서방정 (201006967)
  [0] 가바토파정 100mg: 효능 219자 (시도 1)
  [78] 울트라셋이알서방정: 효능 13자 (시도 1)

최종 수집: 6/6


## STEP 5 — 최종 통합 → pill_info_master.json

`final_mapping` 118종을 순회하면서 효능/용법/주의 본문을 합칩니다:
- **폐업 6종**: `nedrug_results`에서 가져옴
- **나머지 112종**: `results_dtl`에서 XML 파싱한 본문 사용

`clean_val`로 nan/빈 문자열을 None으로 정규화한 뒤 JSON 저장.

**산출물**: `pill_info_master.json` (118종 100% 효능/용법/주의사항 보유)

In [12]:
import math

def clean_val(v):
    """nan/None/빈문자열 → None 변환 (JSON 저장용)."""
    if v is None:
        return None
    if isinstance(v, float) and math.isnan(v):
        return None
    if isinstance(v, str) and not v.strip():
        return None
    return v


OUT_PATH = OUT_DIR / "pill_info_master.json"

final_dict = {}
for _, row in final_mapping.iterrows():
    cidx = int(row['class_idx'])
    seq_str = str(int(row['item_seq'])) if pd.notna(row['item_seq']) else None
    
    # 폐업 6종은 nedrug에서, 나머지는 results_dtl에서
    if cidx in nedrug_results:
        ned = nedrug_results[cidx]
        record = {
            'dl_name':    clean_val(row['dl_name']),
            'dl_company': clean_val(row['dl_company']),
            'item_seq':   seq_str,
            'source':     clean_val(row['source']),
            'etc_otc':    None,
            'chart':      None,
            'material':   None,
            'efficacy':   clean_val(ned['efficacy']),
            'usage':      clean_val(ned['usage']),
            'caution':    clean_val(ned['caution']),
            'storage':    None,
            'image_url':  None,
        }
    else:
        item = results_dtl.get(seq_str, {})
        record = {
            'dl_name':    clean_val(row['dl_name']),
            'dl_company': clean_val(row['dl_company']),
            'item_seq':   seq_str,
            'source':     clean_val(row['source']),
            'etc_otc':    clean_val(item.get('ETC_OTC_CODE')),
            'chart':      clean_val(item.get('CHART')),
            'material':   clean_val(item.get('MATERIAL_NAME')),
            'efficacy':   clean_val(parse_doc_xml(item.get('EE_DOC_DATA', ''))),
            'usage':      clean_val(parse_doc_xml(item.get('UD_DOC_DATA', ''))),
            'caution':    clean_val(parse_doc_xml(item.get('NB_DOC_DATA', ''))),
            'storage':    clean_val(item.get('STORAGE_METHOD')),
            'image_url':  clean_val(item.get('BIG_PRDT_IMG_URL')),
        }
    final_dict[str(cidx)] = record

# 저장
with open(OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(final_dict, f, ensure_ascii=False, indent=2)

# 검증
empty_eff = sum(1 for v in final_dict.values() if not v['efficacy'])
empty_use = sum(1 for v in final_dict.values() if not v['usage'])
empty_cau = sum(1 for v in final_dict.values() if not v['caution'])

print(f"저장: {OUT_PATH}")
print(f"총 {len(final_dict)}종")
print(f"  효능 보유: {len(final_dict) - empty_eff}/118")
print(f"  용법 보유: {len(final_dict) - empty_use}/118")
print(f"  주의 보유: {len(final_dict) - empty_cau}/118")

# 빠진 종 있으면 출력
if empty_eff:
    print(f"\n효능 없는 종:")
    for cidx, v in final_dict.items():
        if not v['efficacy']:
            print(f"  [{cidx}] {v['dl_name']}")

저장: D:\pill_detection\eda\api_data\pill_info_master.json
총 118종
  효능 보유: 118/118
  용법 보유: 118/118
  주의 보유: 118/118
